In [ ]:
instructions_indirect_scripts = '''
You are given a TikTok video, the video has 3 main elements we are interested in:
- Audio:
Use "" to denote text spoken by different characters in the scene. ("This shirt is great!")
Give every speaker an informative name in uppercase letters (SPEAKER A, SPEAKER B , YOUNG GIRL, OLD MAN IN A JACKET,  etc.).
If speech is mumbled or something is unclear then state it as it is.
- Visual:
Describe the overall scene.
Include every character's facial expressions, subtext and emotions.
don't forget to state the various props.
- Captions:
Look for unique caption text that appears on the screen.
no need to repeat subtitles.
all Captions must be mentioned in square brackets.
Different colored subtitles usually mean that there are different speakers in the scene.

return a movie script in the following format and style:
YOUNG GIRL: "Wow, this shirt fits me!"
[NEW HELLO KITTY SHIRT]
a YOUNG GIRL is excited about a shirt that is handed to her by BOY WITH BLUE HAT.
BOY WITH BLUE HAT: "Yeah sure it's a gift"

don't make things up!
state the genre of the TikTok video and if it has sound or not.
make sure the Audio is synchronized with the Visual and the Captions (no need to mark timestamps)
If present, try to mark different speakers using all 3 elements (audio, visual and captions).
don't start your response with "Sure! Here's a movie script of the given TikTok video:" and don't finish if with "Hope that helps"
'''



In [ ]:
import os
from google.cloud import storage
import time
from vertexai.generative_models import (
    GenerationConfig,
    GenerativeModel,
    HarmBlockThreshold,
    HarmCategory,
    Part,
)
import random


In [ ]:
# init model
indirect_extractor = GenerativeModel(
    model_name="gemini-1.5-pro-002",
    system_instruction=[
        instructions_indirect_scripts,
    ],
)


In [ ]:
def list_all_blobs(bucket_name, source_blob_name):
    """Lists all MP4 blobs in the bucket.

    Args:
        bucket_name: The name of the Google Cloud Storage bucket.
        source_blob_name: The path prefix to look for within the bucket.
    """

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix=source_blob_name)
    blob_uris = []

    for blob in blobs:
        if blob.name.lower().endswith(".mp4"):  # Filter for MP4 files
            blob_uri = f"gs://{bucket_name}/{blob.name}"
            blob_uris.append(blob_uri)
        else:
            print(f"Skipping {blob.name} not an mp4 file.")

    return blob_uris



In [ ]:
# ----- Function to generate script from a single video -----
def generate_script_from_video(
    video_uri: str, max_retries=4, retry_delay=40
) -> str:
    """Generates a movie script from a single video URI using Gemini 1.5 Pro.

    Args:
        video_uri: The GCS URI of the video file.
        system_instructions: Instructions on how the model should generate script.
        max_retries: Maximum number of retries if a request fails
        retry_delay: Base delay (in seconds) between retries

    Returns:
        The generated movie script as a string.
    """
    contents = Part.from_uri(uri=video_uri, mime_type="video/mp4")
    prompt = [
        Part.from_text("Video: "),
        contents,
    ]

    for attempt in range(max_retries):
      try:
        response = indirect_extractor.generate_content(prompt,safety_settings=safety)

        if response.text:
            return response.text
        else:
            return f"Error: Could not generate script for {video_uri}, {response.prompt_feedback}"

      except Exception as e:
        print(f"Error generating script for {video_uri} (attempt {attempt + 1}): {e}")
        if attempt < max_retries - 1:
           delay = (retry_delay * (2**attempt)) + random.uniform(0, 1)
           print(f"Retrying in {delay:.2f} seconds...")
           time.sleep(delay)
        else:
            return f"Error: Failed to generate script for {video_uri} after {max_retries} retries."

    return f"Error: Failed to generate script for {video_uri} after {max_retries} retries."

In [ ]:
from google.cloud import storage
import io

def save_string_to_gcs(
    gcs_bucket_name: str,
    gcs_folder_path: str,
    string_content: str,
    file_name: str,
    gcs_client: storage.Client = None  # Allow providing a custom client
) -> bool:
    """
    Saves a string to a text file in a specified Google Cloud Storage (GCS) folder.

    Args:
        gcs_bucket_name: The name of the GCS bucket.
        gcs_folder_path: The path to the folder within the bucket (e.g., "my_folder/subfolder/").
                        Include a trailing slash.
        string_content: The string content to save.
        file_name: The name of the text file to create (e.g., "my_file.txt").
        gcs_client: (Optional) A pre-configured GCS client. If None, a new client is created.

    Returns:
        True if the file was successfully uploaded, False otherwise.
    """

    try:
        # 1. Initialize the GCS client
        if gcs_client is None:
           client = storage.Client()
        else:
            client = gcs_client
        bucket = client.bucket(gcs_bucket_name)

        # 2. Construct the GCS file path
        gcs_file_path = f"{gcs_folder_path}{file_name}"
        blob = bucket.blob(gcs_file_path)

        # 3. Convert the string to bytes and upload
        blob.upload_from_string(string_content, content_type="text/plain")

        print(f"Successfully uploaded {file_name} to GCS at: gs://{gcs_bucket_name}/{gcs_file_path}")
        return True

    except Exception as e:
        print(f"Error uploading to GCS: {e}")
        return False


In [ ]:
# ----- Pipeline function to process multiple videos -----
def process_videos(
    video_uris: list, bucket_name: str, blob_name_prefix: str, request_delay=3
) -> None:
    """Processes a list of video URIs and saves generated scripts to text files.

    Args:
        video_uris: A list of GCS URIs of video files.
        system_instructions: Instructions on how the model should generate script.
        request_delay: seconds to wait between request to the api.
        bucket_name (str): The name of the GCS bucket.
        blob_name_prefix (str): The desired path within the bucket.
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    failures = []
    for video_uri in video_uris:
        print(f"Processing {video_uri}...")
        script = generate_script_from_video(video_uri)

        if script and not script.startswith("Error:"):
            save_string_to_gcs(bucket_name, blob_name_prefix, script, video_uri.split("/")[-1])
        else:
            print(f"  Failed to process: {script}")
            failures.append(video_uri)


        time.sleep(request_delay)
    return failures


In [ ]:
safety = {
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
}

In [ ]:
wtf="https://www.tiktok.com/@tarabellerose/video/7242570820965469442"
'''
Candidate:
{
  "finish_reason": "RECITATION",
  "citation_metadata": {
    "citations": [
      {
        "start_index": 1482,
        "end_index": 1681,
        "uri": "https://www.abc.net.au/news/2023-06-08/how-did-canadian-wildfires-start-smoke-spreading-united-states/102454424"
      },
      {
        "start_index": 1759,
        "end_index": 1923,
        "uri": "https://www.abc.net.au/news/2023-06-08/how-did-canadian-wildfires-start-smoke-spreading-united-states/102454424"
      },
      {
        "start_index": 2036,
        "end_index": 2181,
        "uri": "https://news.cgtn.com/news/2023-06-09/What-are-health-risks-from-wildfire-smoke-1kv4mhxH3EI/index.html"
      }
    ]
  }
}


Token generation stopped because the content potentially contains copyright violations.
'''



content_prohibited = ["https://www.tiktok.com/@charlidamelio/video/7346843466242034974","https://www.tiktok.com/@rclbeauty101/video/7340863270502403370"]
"""
Response:
{
  "prompt_feedback": {
    "block_reason": "PROHIBITED_CONTENT"
  },
  "usage_metadata": {
    "prompt_token_count": 1820,
    "total_token_count": 1820
  },
  "model_version": "gemini-1.5-pro-002"
}
"""

In [ ]:
from typing_extensions import final
# unnote these when ready to go over all videos
# bucket_name = 'influencers_tiktok'
# source_blob_name = "movies/videos/"
# OUTPUT_DIRECTORY = "VideoScripts"

bucket_name = 'influencers_tiktok'
source_blob_name = "movies/videos/"
destination_folder = "movies/videos/VideoScripts/"


# list URIs
uris = list_all_blobs(bucket_name, source_blob_name)
failures = ['gs://influencers_tiktok/movies/videos/@charlidamelio_video_7346843466242034974.mp4', 'gs://influencers_tiktok/movies/videos/@rclbeauty101_video_7340863270502403370.mp4', 'gs://influencers_tiktok/movies/videos/@tarabellerose_video_7242570820965469442.mp4']

# run the functions
logfile = process_videos(uris, bucket_name, destination_folder)
print("Video Processing Complete.")
print("Failed files are saved in logfile")

Skipping movies/videos/VideoScripts/ not an mp4 file.
Skipping movies/videos/Visual2Text/ not an mp4 file.
Skipping movies/videos/Voice2Text/ not an mp4 file.
Processing gs://influencers_tiktok/movies/videos/@alexkay_video_6783398367490854150.mp4...
Successfully uploaded @alexkay_video_6783398367490854150.mp4 to GCS at: gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6783398367490854150.mp4
Processing gs://influencers_tiktok/movies/videos/@alexkay_video_6807140917636648197.mp4...
Successfully uploaded @alexkay_video_6807140917636648197.mp4 to GCS at: gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6807140917636648197.mp4
Processing gs://influencers_tiktok/movies/videos/@alexkay_video_6811970678024195334.mp4...
Successfully uploaded @alexkay_video_6811970678024195334.mp4 to GCS at: gs://influencers_tiktok/movies/videos/VideoScripts/@alexkay_video_6811970678024195334.mp4
Processing gs://influencers_tiktok/movies/videos/@alexkay_video_690587994497963136

In [ ]:
logfile

['gs://influencers_tiktok/movies/videos/@charlidamelio_video_7346843466242034974.mp4',
 'gs://influencers_tiktok/movies/videos/@rclbeauty101_video_7340863270502403370.mp4',
 'gs://influencers_tiktok/movies/videos/@thistrippyhippie_video_7377058339462614304.mp4']

In [ ]:
from typing_extensions import final
# unnote these when ready to go over all videos
# bucket_name = 'influencers_tiktok'
# source_blob_name = "movies/videos/"
# OUTPUT_DIRECTORY = "VideoScripts"

bucket_name = 'influencers_tiktok'
source_blob_name = "movies/test_space/"
destination_folder = "movies/test_space/test_scripts/"


# list URIs
uris = list_all_blobs(bucket_name, source_blob_name)
failures = ['gs://influencers_tiktok/movies/videos/@charlidamelio_video_7346843466242034974.mp4', 'gs://influencers_tiktok/movies/videos/@rclbeauty101_video_7340863270502403370.mp4', 'gs://influencers_tiktok/movies/videos/@tarabellerose_video_7242570820965469442.mp4']

# run the functions
logfile = process_videos(uris, bucket_name, destination_folder)
print("Video Processing Complete.")
print("Failed files are saved in logfile")

Skipping movies/test_space/ not an mp4 file.
Skipping movies/test_space/scripts/ not an mp4 file.
Skipping movies/test_space/test.csv not an mp4 file.
Processing gs://influencers_tiktok/movies/test_space/#duet with @caressmd what a transformation! #skincare #doctorv #transformation #viral #reaction #vitamin [7006755669495893253].mp4...
Successfully uploaded #duet with @caressmd what a transformation! #skincare #doctorv #transformation #viral #reaction #vitamin [7006755669495893253].mp4 to GCS at: gs://influencers_tiktok/movies/test_space/test_scripts/#duet with @caressmd what a transformation! #skincare #doctorv #transformation #viral #reaction #vitamin [7006755669495893253].mp4
Processing gs://influencers_tiktok/movies/test_space/Chamois For the Bois #hike #chamois #colorado #coloradosprings #incline  [7314404563862408494].mp4...
Successfully uploaded Chamois For the Bois #hike #chamois #colorado #coloradosprings #incline  [7314404563862408494].mp4 to GCS at: gs://influencers_tiktok/m

KeyboardInterrupt: 

In [ ]:
logfile

NameError: name 'logfile' is not defined